# Etna Dataset Construction

This notebook builds the Etna case-study dataset for the Cause–Trigger analysis. Waveform features are extracted on an hourly grid, then merged with gas and meteorological context variables. The final dataset is used later for HMML/PCMCI-based trigger analysis.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from obspy.clients.fdsn import Client

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from etna_config import (
    ETNA_WAVEFORM_CONFIG,
    ETNA_GAS_METEO_COLS,
    ETNA_EVENT_TIME,
)

from etna_waveform import build_station_waveform_dataset

from etna_dataset import (
    load_etnagas_csv,
    extract_plume_co2so2_xls,
    create_etna_final_dataset,
)

from etna_plotting_utils import (
    dataset_health_report,
    plot_scaled_dataset,
    compare_station_variable,
    plot_variable_pdfs,
    plot_raw_vs_scaled_pdfs,
    distribution_summary,
    run_teleseismic_checks,
    plot_final_dataset_with_event,
)

client = Client("INGV")
cfg = ETNA_WAVEFORM_CONFIG

## 1. Station and channel metadata

We first inspect the available INGV metadata for the Etna stations. The vertical `EHZ` channel is used for waveform feature extraction because the trigger/response features are based on vertical ground motion.

In [3]:
# identify the station metadata for ME01/ME02, get the exact vertical channel
inventory = client.get_stations(station="ME01" # or "ME02"
                                ,level="response",
                                starttime = "2008-04-12 00:00:00.000",endtime = "2008-05-13 00:00:00.000")
print(inventory)

Inventory created at 2026-05-03T16:04:24.667000Z
	Created by: INGV-ONT WEB SERVICE: fdsnws-station | version: 1.1.64
		    /exist/apps/fdsn-station/fdsnws/station/1/query?starttime=2008-04-...
	Sending institution: eXistDB (INGV-ONT)
	Contains:
		Networks (1):
			IV
		Stations (1):
			IV.ME01 (Mistretta)
		Channels (3):
			IV.ME01..EHZ, IV.ME01..EHN, IV.ME01..EHE


In [4]:
network = inventory[0]
station = network[0] 
num_channels = len(station)
print ('number of channels: ', num_channels)
print(station)

number of channels:  3
Station ME01 (Mistretta)
	Station Code: ME01
	Channel Count: 3/3 (Selected/Total)
	2007-10-18T00:00:00.000000Z - 2009-02-03T23:00:00.000000Z
	Access: open 
	Latitude: 37.9329, Longitude: 14.3611, Elevation: 944.0 m
	Available Channels:
	    ..EH[ZNE]   125.0 Hz  2007-10-18 to 2009-02-03



In [5]:
channel = [c for c in station if c.code == "EHZ"][0]
print(channel)

Channel 'EHZ', Location '' 
	Time range: 2007-10-18T00:00:00.000000Z - 2009-02-03T23:00:00.000000Z
	Latitude: 37.9329, Longitude: 14.3611, Elevation: 944.0 m, Local Depth: 0.0 m
	Azimuth: 0.00 degrees from north, clockwise
	Dip: -90.00 degrees down from horizontal
	Sampling Rate: 125.00 Hz
	Sensor (Description): None (LENNARTZ LE3D-5S)
	Response information available


## 2. Hourly waveform feature extraction

For each station, waveform data are processed in daily chunks with padding for filter stability. Three log-transformed RMS features are extracted:

- `T_log`: low-frequency teleseismic trigger band, aggregated by hourly maximum.
- `S_log`: background/state band, aggregated by hourly mean.
- `Y_log`: high-frequency response band, aggregated by hourly maximum.

In [ ]:
me01_wave, me01_failures = build_station_waveform_dataset(
    client=client,
    station="ME01",
    cfg=cfg,
)

me02_wave, me02_failures = build_station_waveform_dataset(
    client=client,
    station="ME02",
    cfg=cfg,
)

c:\Users\cescedes\anaconda3\envs\M\Lib\site-packages\obspy\core\inventory\response.py:988: UserWarning: Input sampling rate of stage 3 is inconsistent with the previous stages' output sampling rate
  warnings.warn(msg % i)


ME01 OK  2008-04-12


c:\Users\cescedes\anaconda3\envs\M\Lib\site-packages\obspy\core\inventory\response.py:988: UserWarning: Input sampling rate of stage 3 is inconsistent with the previous stages' output sampling rate
  warnings.warn(msg % i)


ME01 OK  2008-04-13


c:\Users\cescedes\anaconda3\envs\M\Lib\site-packages\obspy\core\inventory\response.py:988: UserWarning: Input sampling rate of stage 3 is inconsistent with the previous stages' output sampling rate
  warnings.warn(msg % i)


...

c:\Users\cescedes\anaconda3\envs\M\Lib\site-packages\obspy\core\inventory\response.py:988: UserWarning: Input sampling rate of stage 3 is inconsistent with the previous stages' output sampling rate
  warnings.warn(msg % i)


ME02 OK  2008-04-18


c:\Users\cescedes\anaconda3\envs\M\Lib\site-packages\obspy\core\inventory\response.py:988: UserWarning: Input sampling rate of stage 3 is inconsistent with the previous stages' output sampling rate
  warnings.warn(msg % i)


ME02 OK  2008-04-19


c:\Users\cescedes\anaconda3\envs\M\Lib\site-packages\obspy\core\inventory\response.py:988: UserWarning: Input sampling rate of stage 3 is inconsistent with the previous stages' output sampling rate
  warnings.warn(msg % i)


## 3. Save waveform checkpoints

The waveform extraction step is computationally expensive. The extracted hourly waveform features are saved as checkpoints so later steps can be rerun without downloading and processing the waveform data again.

In [ ]:
# checkpoint
me01_wave.to_pickle("../etna_data/etna_me01_waveform.pkl")
me02_wave.to_pickle("../etna_data/etna_me02_waveform.pkl")

In [ ]:
# load from checkpoint
#me01_wave = pd.read_pickle("../etna_data/etna_me01_waveform.pkl")
#me02_wave = pd.read_pickle("../etna_data/etna_me02_waveform.pkl")

In [ ]:
# also save as CSV for easier inspection
#me01_wave.reset_index().to_csv("../etna_data/etna_me01_waveform.csv", index=False)
#me02_wave.reset_index().to_csv("../etna_data/etna_me02_waveform.csv", index=False)

## 4. Gas and meteorological context variables

We add slower contextual variables from ETNAGAS and plume observations. Rain is excluded from the first-pass dataset because it is sparse and near-constant in short event windows. These variables are merged onto the hourly waveform grid.

- WindSpeed 
- Patm_3 
- AirTemp_3 
- CO2_3 
- plume data (SO2/CO2 ratio)

In [ ]:
plume_df = extract_plume_co2so2_xls("../etna_data/1012-1_VolcanicGas_Etna.xls")

In [ ]:
etnagas_df = load_etnagas_csv(
    path="../etna_data/3c.csv",
    value_cols=ETNA_GAS_METEO_COLS,
)

display(etnagas_df.head())
display(etnagas_df.isna().mean().sort_values())

## 5. Merge, scale, and save final hourly datasets

The waveform, gas, meteorological, and plume variables are merged by timestamp. Raw variables are retained, and scaled versions are produced for causal discovery. The output is saved separately for ME01 and ME02.

In [ ]:
final_me01_raw, final_me01_scaled = create_etna_final_dataset(
    wave_df=me01_wave,
    station_name="ME01",
    out_csv="../etna_data/FINAL_ME01.csv",
    etnagas_df=etnagas_df,
    etnagas_cols=ETNA_GAS_METEO_COLS,
    plume_df=plume_df,
)

final_me02_raw, final_me02_scaled = create_etna_final_dataset(
    wave_df=me02_wave,
    station_name="ME02",
    out_csv="../etna_data/FINAL_ME02.csv",
    etnagas_df=etnagas_df,
    etnagas_cols=ETNA_GAS_METEO_COLS,
    plume_df=plume_df,
)

In [ ]:
# load from checkpoint
#final_me01 = pd.read_pickle("../etna_data/final_me01.pkl")
#final_me02 = pd.read_pickle("../etna_data/final_me02.pkl")

## 6. Dataset quality checks

We check dataset size, timestamp order, duplicate timestamps, missing values, and variable distributions. These checks ensure that the final hourly dataset is suitable for causal discovery.

In [ ]:
station_data = {
    "ME01": {
        "wave": me01_wave,
        "final_raw": final_me01_raw,
        "final_scaled": final_me01_scaled,
    },
    "ME02": {
        "wave": me02_wave,
        "final_raw": final_me02_raw,
        "final_scaled": final_me02_scaled,
    },
}

In [ ]:
for sta, d in station_data.items():
    dataset_health_report(d["final_raw"], f"{sta} hourly raw")
    dataset_health_report(d["final_scaled"], f"{sta} hourly scaled")

## 7. Time-series inspection

The scaled variables are plotted to inspect temporal structure, outliers, and station-level differences. This helps identify whether the extracted variables behave consistently across ME01 and ME02.

#### Plot scaled variables for both stations

In [ ]:
for sta, d in station_data.items():
    plot_scaled_dataset(d["final_scaled"], sta)

#### Compare ME01 and ME02 directly in one figure

In [ ]:
for var in ["S_log_scaled", "T_log_scaled", "Y_log_scaled"]:
    if var in final_me01_scaled.columns and var in final_me02_scaled.columns:
        compare_station_variable(final_me01_scaled, final_me02_scaled, var)


## 8. Distribution diagnostics

Empirical density plots and summary statistics are used to inspect skewness, outliers, and scaling behavior. These diagnostics justify the preprocessing choices before applying HMML or PCMCI.

#### PDF / distribution plots

In [ ]:
plot_variable_pdfs(final_me01_scaled, "ME01 hourly scaled")
plot_variable_pdfs(final_me02_scaled, "ME02 hourly scaled")

#### raw-vs-scaled PDFs

In [ ]:
variable_pairs = [
    ("S_log", "S_log_scaled"),
    ("T_log", "T_log_scaled"),
    ("Y_log", "Y_log_scaled"),
    ("CO2_3", "CO2_3_scaled"),
    ("AirTemp_3", "AirTemp_3_scaled"),
    ("Patm_3", "Patm_3_scaled"),
    ("WindSpeed", "WindSpeed_scaled"),
    ("CO2_SO2", "CO2_SO2_scaled"),
]

plot_raw_vs_scaled_pdfs(
    final_me01_raw,
    final_me01_scaled,
    "ME01 hourly",
    variable_pairs,
)

plot_raw_vs_scaled_pdfs(
    final_me02_raw,
    final_me02_scaled,
    "ME02 hourly",
    variable_pairs,
)

#### distribution summary table

In [ ]:
summary_me01 = distribution_summary(final_me01_scaled, "ME01 hourly scaled")
summary_me02 = distribution_summary(final_me02_scaled, "ME02 hourly scaled")

display(summary_me01)
display(summary_me02)

## 9. Teleseismic arrival diagnostics

We inspect the waveform around the Wenchuan earthquake arrival. The 1-minute RMS is compared with hourly maximum aggregation to verify that the hourly grid preserves the arrival-period energy. The spectrogram confirms that the selected frequency bands capture the relevant trigger and response components.

In [ ]:
results = {}

for station in ["ME01", "ME02"]:
    results[station] = run_teleseismic_checks(
        client=client,
        station=station,
        cfg=ETNA_WAVEFORM_CONFIG,
        event_time=ETNA_EVENT_TIME,
    )

## 10. Final hourly dataset with event marker

The final scaled dataset is plotted with the teleseismic event time marked. This provides a visual check that the causal-analysis dataset aligns with the known event window.

In [ ]:
plot_final_dataset_with_event(
    "../etna_data/FINAL_ME01_scaled.csv",
    station="ME01",
    event_time=ETNA_EVENT_TIME,
)

plot_final_dataset_with_event(
    "../etna_data/FINAL_ME02_scaled.csv",
    station="ME02",
    event_time=ETNA_EVENT_TIME,
)